# Forest Impact Simulator (Python)

Python port of the [Forest Impact Simulator](https://github.com/karimogit/Forest-Impact-Simulator) web app.

Calculations (growth curves, clear-cutting carbon, biodiversity/resilience scales, water/air percentages, planting timelines) are aligned with the original TypeScript implementation.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Ensure the package is importable when the notebook runs from the repo root
repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
import matplotlib.pyplot as plt

from forest_impact import ForestImpactSimulator, TREE_TYPES, get_tree_by_name, format_area
from forest_impact.export import write_exports, generate_csv
from forest_impact.planting import calculate_planting_timeline

simulator = ForestImpactSimulator()
print(f"Simulator v{simulator.version} ready — {len(TREE_TYPES)} tree species loaded")

## 2. Load forest plot data

Supported input columns: `plot_id`, `area` (ha), `tree_type`, `tree_count`, `tree_density`, `latitude`, `longitude`.

You can also load the comprehensive one-row export CSV (`sample-forest-planting-data.csv`).

In [ ]:
def load_plot_csv(path: str):
    df = pd.read_csv(path)
    # Comprehensive single-row export format
    if "area_hectares" in df.columns and "total_trees" in df.columns:
        row = df.iloc[0]
        plots = [{
            "plot_id": "Imported",
            "area": float(row.get("area_hectares") or 0),
            "tree_type": str(row.get("tree_names") or "Oak").split(";")[0],
            "tree_count": int(row.get("total_trees") or 0),
            "tree_density": float(row.get("density_trees_hectare") or 0),
            "latitude": float(row["latitude"]) if pd.notna(row.get("latitude")) else None,
            "longitude": float(row["longitude"]) if pd.notna(row.get("longitude")) else None,
        }]
        meta = {
            "years": int(row.get("simulation_years") or 50),
            "temperature": float(row["temperature_c"]) if pd.notna(row.get("temperature_c")) else None,
            "precipitation": float(row["precipitation_mm"]) if pd.notna(row.get("precipitation_mm")) else None,
            "soil_carbon": float(row["soil_carbon_g_kg"]) if pd.notna(row.get("soil_carbon_g_kg")) else None,
            "spacing": float(row["spacing_meters"]) if pd.notna(row.get("spacing_meters")) else None,
            "region": {
                "north": float(row["region_north"]) if pd.notna(row.get("region_north")) else None,
                "south": float(row["region_south"]) if pd.notna(row.get("region_south")) else None,
                "east": float(row["region_east"]) if pd.notna(row.get("region_east")) else None,
                "west": float(row["region_west"]) if pd.notna(row.get("region_west")) else None,
            },
        }
        return plots, meta

    plots = []
    for i, row in df.iterrows():
        plots.append({
            "plot_id": str(row.get("plot_id", f"Plot_{i+1:03d}")),
            "area": float(row.get("area", 0) or 0),
            "tree_type": str(row.get("tree_type", "Oak")),
            "tree_count": int(row.get("tree_count", 0) or 0),
            "tree_density": float(row.get("tree_density", 0) or 0),
            "latitude": float(row["latitude"]) if "latitude" in df.columns and pd.notna(row.get("latitude")) else None,
            "longitude": float(row["longitude"]) if "longitude" in df.columns and pd.notna(row.get("longitude")) else None,
        })
    return plots, {"years": 50}


plots_data, import_meta = load_plot_csv("sample-forest-planting-data.csv")
display(pd.DataFrame(plots_data))
print("Import meta:", {k: v for k, v in import_meta.items() if k != "region"})

## 3. Configure and run simulation

Set `mode` to `planting` or `clear-cutting`. For clear-cutting, set `average_tree_age`.

In [ ]:
MODE = "planting"          # or "clear-cutting"
YEARS = import_meta.get("years", 50)
AVERAGE_TREE_AGE = 20      # used in clear-cutting mode

results = simulator.simulate_from_plots(
    plots_data,
    years=YEARS,
    mode=MODE,
    average_tree_age=AVERAGE_TREE_AGE,
    temperature=import_meta.get("temperature"),
    precipitation=import_meta.get("precipitation"),
    soil_carbon=import_meta.get("soil_carbon"),
)

# Prefer imported region / spacing when available
region = import_meta.get("region") or {}
if all(region.get(k) is not None for k in ("north", "south", "east", "west")):
    results["metadata"]["location"]["region"] = region
if import_meta.get("spacing"):
    results["plantingData"]["spacing"] = import_meta["spacing"]

impact = results["impactResults"]
planting = results["plantingData"]

print(f"Mode: {MODE} | Years: {YEARS}")
print(f"Area: {format_area(planting['area'])} | Trees: {planting['totalTrees']:,}")
print(f"Annual carbon (period average): {impact['carbonSequestration']:,.1f} kg CO₂/year")
print(f"Total carbon over {YEARS} years: {impact['totalCarbon']:,.1f} kg CO₂")
print(f"Biodiversity: {impact['biodiversityImpact']:.1f}/5")
print(f"Resilience: {impact['forestResilience']:.1f}/5")
print(f"Water retention: {impact['waterRetention']:.0f}%")
print(f"Air quality: {impact['airQualityImprovement']:.0f}%")
print(f"Social impact: {results['socialImpact']:.1f}/5")
print("Comparisons:", {k: round(v, 1) for k, v in results['comparisons'].items()})
if impact.get("clearCuttingBreakdown"):
    print("Clear-cutting breakdown:", impact["clearCuttingBreakdown"])

## 4. Visualize impacts

In [ ]:
labels = ["Biodiversity", "Resilience", "Water %", "Air %"]
values = [
    impact["biodiversityImpact"],
    impact["forestResilience"],
    impact["waterRetention"] / 20,  # scale 0-95% toward ~0-5 for charting
    max(0, impact["airQualityImprovement"]) / 20,
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(labels, values, color=["#2d6a4f", "#40916c", "#52b788", "#74c69d"])
axes[0].set_ylim(0, 5.5)
axes[0].set_title("Impact scores (water/air scaled ÷20)")
axes[0].set_ylabel("Score")

years = list(range(1, YEARS + 1))
from forest_impact.growth import get_planting_growth_factor
mature = impact["matureAnnualCarbon"]
cumulative = []
running = 0.0
for y in years:
    running += mature * get_planting_growth_factor(y)
    cumulative.append(running / 1000)  # metric tons
axes[1].plot(years, cumulative, color="#1b4332", linewidth=2)
axes[1].set_title("Cumulative carbon (growth curve)")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Metric tons CO₂")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Export results

Exports match the original web app formats: comprehensive CSV, JSON, and GeoJSON.

In [ ]:
from datetime import datetime

prefix = f"forest_impact_results_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"
paths = write_exports(results, prefix)
print("Wrote:")
for kind, path in paths.items():
    print(f"  {kind}: {path}")

print("\nCSV preview:")
print(generate_csv(results))

## 6. Quick clear-cutting example

In [ ]:
clear_results = simulator.simulate(
    trees=["Oak"],
    total_trees=planting["totalTrees"],
    area_hectares=planting["area"],
    years=YEARS,
    mode="clear-cutting",
    latitude=results["metadata"]["location"]["latitude"],
    longitude=results["metadata"]["location"]["longitude"],
    average_tree_age=AVERAGE_TREE_AGE,
    temperature=import_meta.get("temperature"),
)
cc = clear_results["impactResults"]["clearCuttingBreakdown"]
print(f"Immediate release: {cc['immediate']:,.1f} kg CO₂")
print(f"Lost future sequestration: {cc['lost_future']:,.1f} kg CO₂")
print(f"Total emissions: {cc['total']:,.1f} kg CO₂")
print(f"Air quality impact: {clear_results['impactResults']['airQualityImprovement']:.0f}%")